# 04 – GPU COLMAP via Docker




In [ ]:
%%bash
set -e
BASE=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025


docker run --rm --gpus all nvidia/cuda:12.2.0-base-ubuntu22.04 nvidia-smi


Wed Dec 24 06:38:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8             13W /   80W |     174MiB /   6144MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Rebuild pipeline



In [2]:
%%bash
set -e
BASE=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025
OUT=$BASE/outputs/ETs_cluster_0001
mkdir -p "$OUT" "$OUT/sparse"
rm -f "$OUT/colmap.db" "$OUT/dense_points.ply"
rm -rf "$OUT/sparse/0" "$OUT/dense"

# 1) Feature extraction
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap feature_extractor     --database_path outputs/ETs_cluster_0001/colmap.db     --image_path outputs/ETs_cluster_0001/images     --ImageReader.single_camera 1     --ImageReader.camera_model SIMPLE_RADIAL

# 2) Exhaustive matcher
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap exhaustive_matcher     --database_path outputs/ETs_cluster_0001/colmap.db

# 3) Sparse reconstruction
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap mapper     --database_path outputs/ETs_cluster_0001/colmap.db     --image_path outputs/ETs_cluster_0001/images     --output_path outputs/ETs_cluster_0001/sparse     --Mapper.ba_refine_principal_point 0


rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/frames.bin': Permission denied
rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/points3D.bin': Permission denied
rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/cameras.bin': Permission denied
rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/images.bin': Permission denied
rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/rigs.bin': Permission denied
rm: cannot remove '/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/ETs_cluster_0001/sparse/0/project.ini': Permission denied
rm: cannot remove '/home/shamika/documents/p

CalledProcessError: Command 'b'set -e\nBASE=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025\nOUT=$BASE/outputs/ETs_cluster_0001\nmkdir -p "$OUT" "$OUT/sparse"\nrm -f "$OUT/colmap.db" "$OUT/dense_points.ply"\nrm -rf "$OUT/sparse/0" "$OUT/dense"\n\n# 1) Feature extraction\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap feature_extractor     --database_path outputs/ETs_cluster_0001/colmap.db     --image_path outputs/ETs_cluster_0001/images     --ImageReader.single_camera 1     --ImageReader.camera_model SIMPLE_RADIAL\n\n# 2) Exhaustive matcher\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap exhaustive_matcher     --database_path outputs/ETs_cluster_0001/colmap.db\n\n# 3) Sparse reconstruction\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap mapper     --database_path outputs/ETs_cluster_0001/colmap.db     --image_path outputs/E

In [3]:
%%bash
set -e
BASE=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025

# 4) Undistort for dense stage
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap image_undistorter     --image_path outputs/ETs_cluster_0001/images     --input_path outputs/ETs_cluster_0001/sparse/0     --output_path outputs/ETs_cluster_0001/dense     --output_type COLMAP

# 5) GPU PatchMatch stereo
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap patch_match_stereo     --workspace_path outputs/ETs_cluster_0001/dense     --workspace_format COLMAP     --PatchMatchStereo.geom_consistency true     --PatchMatchStereo.gpu_index 0

# 6) Fuse depth maps to dense point cloud
docker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap stereo_fusion     --workspace_path outputs/ETs_cluster_0001/dense     --workspace_format COLMAP     --input_type geometric     --output_path outputs/ETs_cluster_0001/dense_points.ply

ls -l outputs/ETs_cluster_0001/dense_points.ply



== CUDA ==

CUDA Version 12.9.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.



I20251224 06:39:06.827397     1 misc.cc:44] 
Reading reconstruction
I20251224 06:39:06.834450     1 image.cc:374] => Reconstruction with 7 images and 861 points
I20251224 06:39:06.834563     1 misc.cc:44] 
Image undistortion
I20251224 06:39:06.836946     1 undistortion.cc:197] Undistorting image [1/7]
I20251224 06:39:06.963896     1 undistortion.cc:197] Undistorting image [2/7]
I20251224 06:39:06.988756     1 undistortion.cc:197] Undistorting image [3/7]
I20251224 06:39:06.989823     1 undistortion.cc:197] Undistorting image [4/7]
I20251224 06:39:06.989887     1 undistortion.cc:197] Undistorting image [5/7]
I20251224 06:39:06.989899     1 undistortion.cc:197] Undistorting image [6/7]
I20251224 06:39:06.989907     1 undistortion.cc:197] Undistorting image [7/7]
I20251224 06:39:06.989912     1 undistortion.cc:207] Writing reconstruction...
I20251224 06:39:07.105237     1 undistortion.cc:212] Writing configuration...
I20251224 06:39:07.106158     1 undistortion.cc:216] Writing scripts...



== CUDA ==

CUDA Version 12.9.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.



I20251224 06:39:10.537935     1 patch_match.cc:205] Reading workspace...
I20251224 06:39:10.541560     1 patch_match.cc:235] Reading configuration...
I20251224 06:39:10.542542     1 patch_match.cc:366] Configuration has 7 problems...
I20251224 06:39:10.543545     1 timer.cc:90] Elapsed time: 0.000 [minutes]



== CUDA ==

CUDA Version 12.9.1

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.



I20251224 06:39:14.869063     1 misc.cc:51] 
StereoFusion::Options
---------------------
I20251224 06:39:14.869201     1 fusion.cc:77] mask_path: 
I20251224 06:39:14.869251     1 fusion.cc:78] max_image_size: -1
I20251224 06:39:14.869258     1 fusion.cc:79] min_num_pixels: 5
I20251224 06:39:14.869263     1 fusion.cc:80] max_num_pixels: 10000
I20251224 06:39:14.869269     1 fusion.cc:81] max_traversal_depth: 100
I20251224 06:39:14.869274     1 fusion.cc:82] max_reproj_error: 2
I20251224 06:39:14.869648     1 fusion.cc:83] max_depth_error: 0.01
I20251224 06:39:14.869676     1 fusion.cc:84] max_normal_error: 10
I20251224 06:39:14.869683     1 fusion.cc:85] check_num_images: 50
I20251224 06:39:14.869688     1 fusion.cc:86] use_cache: 0
I20251224 06:39:14.869692     1 fusion.cc:87] cache_size: 32
I20251224 06:39:14.869699     1 fusion.cc:90] bbox_min: -3.40282e+38 -3.40282e+38 -3.40282e+38
I20251224 06:39:14.869832     1 fusion.cc:91] bbox_max: 3.40282e+38 3.40282e+38 3.40282e+38
I20251224 

CalledProcessError: Command 'b'set -e\nBASE=/home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025\n\n# 4) Undistort for dense stage\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap image_undistorter     --image_path outputs/ETs_cluster_0001/images     --input_path outputs/ETs_cluster_0001/sparse/0     --output_path outputs/ETs_cluster_0001/dense     --output_type COLMAP\n\n# 5) GPU PatchMatch stereo\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap patch_match_stereo     --workspace_path outputs/ETs_cluster_0001/dense     --workspace_format COLMAP     --PatchMatchStereo.geom_consistency true     --PatchMatchStereo.gpu_index 0\n\n# 6) Fuse depth maps to dense point cloud\ndocker run --rm --gpus all -v "$BASE":/workspace -w /workspace colmap/colmap:latest   colmap stereo_fusion     --workspace_path outputs/ETs_cluster_0001/dense     --workspace_format COLMAP     --input_type geom